# 02 — Advanced Retrieval

El Naive RAG tiene un problema fundamental: **vocabulary mismatch**.
Si preguntas de una manera y el paper lo explica de otra, no lo encuentra.

Este notebook implementa 3 técnicas que atacan este problema:

| Técnica | Idea central |
|---------|-------------|
| **HyDE** | Genera un doc hipotético que respondería la pregunta, busca con él |
| **RAG-Fusion** | Genera 4 variantes de la pregunta, fusiona resultados con RRF |
| **CrossEncoder** | Reordena los candidatos con un modelo más preciso |

**Prerequisito**: haber ejecutado el notebook 01 (el vector store debe estar creado)

In [1]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / '.env')

from src.ingestion.embedder import get_embeddings
from src.retrieval.vectorstore import load_vectorstore
from src.generation.chain import build_rag_chain, format_docs

embeddings = get_embeddings()
vectorstore = load_vectorstore(embeddings)
print('Vector store cargado')

/Users/nicoglez/.pyenv/versions/3.12.13/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


Cargando modelo de embeddings: BAAI/bge-m3
(Primera vez: descarga ~570 MB — después va al instante)
Vector store cargado


## Técnica 1 — HyDE (Hypothetical Document Embeddings)

**El problema**: el embedding de una pregunta corta es muy distinto al embedding
de un párrafo técnico que la respondería. Buscamos preguntas, pero el índice
tiene respuestas.

**La solución HyDE**:
1. Pedimos al LLM: *"escribe un párrafo que respondería esta pregunta"*
2. Ese párrafo hipotético se parece más (en embedding) a los chunks reales
3. Buscamos con el párrafo, no con la pregunta

In [2]:
from src.retrieval.strategies import retrieve_hyde

question = 'What are the limitations of multi-head attention?'

print('=== Naive RAG ===')
naive_docs = vectorstore.similarity_search(question, k=3)
for i, doc in enumerate(naive_docs):
    print(f'[{i+1}] {doc.metadata["source"]} | {doc.page_content[:150]}...')

print('\n=== HyDE ===')
hyde_docs = retrieve_hyde(vectorstore, question, k=3)
for i, doc in enumerate(hyde_docs):
    print(f'[{i+1}] {doc.metadata["source"]} | {doc.page_content[:150]}...')

=== Naive RAG ===
[1] attention_is_all_you_need | output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows th...
[2] attention_is_all_you_need | is similar to that of single-head attention with full dimensionality.
3.2.3 Applications of Attention in our Model
The Transformer uses multi-head att...
[3] llama2 | Grouped-Query Attention. A standard practice for autoregressive decoding is to cache the key (K) and
value (V) pairs for the previous tokens in the se...

=== HyDE ===
[HyDE] Documento hipotético generado (903 chars)
[1] attention_is_all_you_need | output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows th...
[2] attention_is_all_you_need | much faster and more space-efficient in practice, since it can be implemented using highly optimized
matrix multiplication code.
While for small value...
[3] atte

In [3]:
# Genera respuesta con HyDE
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
import os

RAG_PROMPT = ChatPromptTemplate.from_template("""
Responde basándote ÚNICAMENTE en el contexto. Si no está en el contexto, dilo.

Contexto:
{context}

Pregunta: {question}
Respuesta:""")

llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0, api_key=os.getenv('GROQ_API_KEY'))

hyde_context = format_docs(hyde_docs)
hyde_answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({'context': hyde_context, 'question': question})

print(f'Q: {question}')
print(f'\nA (HyDE): {hyde_answer}')

Q: What are the limitations of multi-head attention?

A (HyDE): No está en el contexto. El contexto proporcionado describe la arquitectura y el funcionamiento de la atención multi-cabeza (multi-head attention) en el modelo Transformer, pero no menciona explícitamente las limitaciones de esta técnica.


## Técnica 2 — RAG-Fusion (Multi-Query + RRF)

**El problema**: una sola query tiene un punto de vista limitado. Puede que
la información esté en el índice pero formulada de otra manera.

**La solución RAG-Fusion**:
1. Genera 4 variantes de la pregunta con el LLM
2. Busca con cada una → 4 listas de resultados
3. **Reciprocal Rank Fusion**: un chunk que aparece en múltiples listas
   sube en el ranking final (es una señal fuerte de relevancia)

In [4]:
from src.retrieval.strategies import retrieve_rag_fusion

question = 'How does BERT differ from GPT in training objectives?'

print('=== RAG-Fusion ===')
fusion_docs = retrieve_rag_fusion(vectorstore, question, k=4)

print(f'\n{len(fusion_docs)} chunks recuperados (post-fusión):')
for i, doc in enumerate(fusion_docs):
    print(f'[{i+1}] {doc.metadata["source"]} | p.{doc.metadata["page"]} | {doc.page_content[:180]}...')
    print()

=== RAG-Fusion ===
[RAG-Fusion] 4 queries generadas

4 chunks recuperados (post-fusión):
[1] bert | p.13 | ture differences, BERT and OpenAI GPT are ﬁne-
tuning approaches, while ELMo is a feature-based
approach.
The most comparable existing pre-training
method to BERT is OpenAI GPT, wh...

[2] bert | p.13 | between how BERT and GPT were trained:
• GPT is trained on the BooksCorpus (800M
words); BERT is trained on the BooksCor-
pus (800M words) and Wikipedia (2,500M
words).
• GPT uses ...

[3] gpt3 | p.17 | BERT baseline from the original paper but is still well below both human performance and state-of-the-art approaches
which augment neural networks with symbolic systems [RLL+19]. O...

[4] bert | p.12 | BERT (Ours)
Trm Trm Trm
Trm Trm Trm
...
...
Trm Trm Trm
Trm Trm Trm
...
...
OpenAI GPT
Lstm
ELMo
Lstm Lstm
Lstm Lstm Lstm
Lstm Lstm Lstm
Lstm Lstm Lstm
 T1 T2  TN...
...
...
...
.....



In [5]:
# Respuesta con RAG-Fusion
fusion_context = format_docs(fusion_docs)
fusion_answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({'context': fusion_context, 'question': question})

print(f'Q: {question}')
print(f'\nA (RAG-Fusion): {fusion_answer}')

Q: How does BERT differ from GPT in training objectives?

A (RAG-Fusion): No hay información en el contexto sobre las diferencias en los objetivos de entrenamiento entre BERT y GPT. El contexto solo menciona las diferencias en la arquitectura, el conjunto de datos de entrenamiento y los hiperparámetros, pero no proporciona información sobre los objetivos de entrenamiento.


## Técnica 3 — CrossEncoder Reranking

**El problema**: la similitud coseno compara vectores por separado (pregunta y
chunk en aislamiento). Es rápido pero poco preciso.

**La solución CrossEncoder**:
- Un CrossEncoder evalúa el par *(pregunta, chunk)* **juntos**
- Ve la interacción entre ambos textos → mucho más preciso
- Tradeoff: más lento (no se puede indexar, hay que evaluar cada par)

**Estrategia**: recuperar más candidatos con coseno (rápido) y reordenarlos
con CrossEncoder (preciso). Lo mejor de ambos mundos.

In [6]:
from src.retrieval.strategies import retrieve_with_reranking

question = 'What is chain-of-thought prompting and when does it work best?'

print('=== Naive (top 4 por coseno) ===')
naive_docs = vectorstore.similarity_search(question, k=4)
for i, doc in enumerate(naive_docs):
    print(f'[{i+1}] {doc.metadata["source"]} | {doc.page_content[:150]}...')

print('\n=== CrossEncoder Reranking ===')
reranked_docs = retrieve_with_reranking(vectorstore, question, k_initial=10, k_final=4)
for i, doc in enumerate(reranked_docs):
    print(f'[{i+1}] {doc.metadata["source"]} | {doc.page_content[:150]}...')

=== Naive (top 4 por coseno) ===
[1] chain_of_thought | While chain-of-thought prompting is in principle applicable for any text-to-text task, it is more
helpful for some tasks than others. Based on the exp...
[2] chain_of_thought | C Extended Related Work
Chain-of-thought prompting is a general approach that is inspired by several prior directions: prompt-
ing, natural language e...
[3] chain_of_thought | where each seed had a different randomly shufﬂed order of exemplars. As LaMDA experiments
did not show large variance among different seeds, to save c...
[4] chain_of_thought | 2022, inter alia)).
Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning
in language models.
1. Firs...

=== CrossEncoder Reranking ===
[Reranking] 10 candidatos → reranking con CrossEncoder


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

[1] chain_of_thought | While chain-of-thought prompting is in principle applicable for any text-to-text task, it is more
helpful for some tasks than others. Based on the exp...
[2] chain_of_thought | 2022, inter alia)).
Chain-of-thought prompting has several attractive properties as an approach for facilitating reasoning
in language models.
1. Firs...
[3] chain_of_thought | (Cobbe et al., 2021), chain-of-thought prompting with PaLM 540B outperforms standard prompting
by a large margin and achieves new state-of-the-art per...
[4] chain_of_thought | C Extended Related Work
Chain-of-thought prompting is a general approach that is inspired by several prior directions: prompt-
ing, natural language e...


In [7]:
# Respuesta con CrossEncoder
reranked_context = format_docs(reranked_docs)
reranked_answer = (RAG_PROMPT | llm | StrOutputParser()).invoke({'context': reranked_context, 'question': question})

print(f'Q: {question}')
print(f'\nA (CrossEncoder): {reranked_answer}')

Q: What is chain-of-thought prompting and when does it work best?

A (CrossEncoder): Chain-of-thought prompting es un enfoque que permite a los modelos de lenguaje descomponer problemas en pasos intermedios, lo que facilita el razonamiento en tareas que requieren múltiples pasos. Funciona mejor cuando se cumplen tres condiciones: (1) la tarea es desafiante y requiere razonamiento, (2) no se especifican las otras dos condiciones en el contexto proporcionado. 

En general, se puede decir que chain-of-thought prompting es útil para tareas que requieren razonamiento y descomposición de problemas en pasos intermedios, como problemas de matemáticas o tareas que requieren múltiples pasos para resolver.


## Comparación visual rápida

Misma pregunta, tres estrategias de retrieval:

In [8]:
question = 'What are the advantages of sparse mixture of experts models?'

strategies = {
    'Naive': vectorstore.similarity_search(question, k=4),
    'HyDE': retrieve_hyde(vectorstore, question, k=4),
    'RAG-Fusion': retrieve_rag_fusion(vectorstore, question, k=4),
    'CrossEncoder': retrieve_with_reranking(vectorstore, question, k_initial=10, k_final=4),
}

print(f'Pregunta: "{question}"\n')
for name, docs in strategies.items():
    sources = [d.metadata['source'] for d in docs]
    print(f'{name:15} → {sources}')

[HyDE] Documento hipotético generado (921 chars)
[RAG-Fusion] 4 queries generadas
[Reranking] 10 candidatos → reranking con CrossEncoder
Pregunta: "What are the advantages of sparse mixture of experts models?"

Naive           → ['mixtral', 'mixtral', 'mixtral', 'mixtral']
HyDE            → ['mixtral', 'rag_survey', 'gpt3', 'bert']
RAG-Fusion      → ['mixtral', 'mixtral', 'mixtral', 'llama2']
CrossEncoder    → ['mixtral', 'mixtral', 'mixtral', 'mixtral']


## Resumen

| Técnica | Ventaja | Desventaja |
|---------|---------|------------|
| Naive | Rápido, simple | Vocabulary mismatch |
| HyDE | Mejor con preguntas abstractas | +1 llamada LLM |
| RAG-Fusion | Más robusto, multi-perspectiva | +4 llamadas LLM |
| CrossEncoder | Más preciso en relevancia | Más lento |

→ En el **notebook 03** medimos cuál mejora realmente con RAGAS.